In [67]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
) 
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay

import mlflow

In [68]:
from pathlib import Path

DATA_PATH = Path("./Data")
RAW_DATA_PATH = DATA_PATH / "raw" / "Fraud_Data.csv"
PROCESSED_DATA_PATH = DATA_PATH / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [69]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Fraud Detection") 

<Experiment: artifact_location='file:C:/Users/trixr/Desktop/FraudDetection/mlflow-data/artifacts/1', creation_time=1781179797941, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781179797941, lifecycle_stage='active', name='Fraud Detection', tags={}, trace_location=None, workspace='default'>

In [70]:
import mlflow.sklearn 

mlflow.xgboost.autolog()
mlflow.lightgbm.autolog()
mlflow.sklearn.autolog()

# Load Data

In [71]:
X_train = pd.read_csv(PROCESSED_DATA_PATH / "4.1_X_train.csv")
y_train = pd.read_csv(PROCESSED_DATA_PATH / "4.2_y_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_PATH / "4.3_X_test.csv")
y_test = pd.read_csv(PROCESSED_DATA_PATH / "4.4_y_test.csv")

In [72]:
X_train.head()

,purchase_value,source,browser,sex,age,time_velocity,ip_user_share_count,device_user_share_count,day,hour,...,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M,source_fr_enc,browser_fr_enc,source_targ_enc,browser_targ_enc
0,14,Ads,Chrome,F,38,7212744.0,0,0,25,11,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
1,14,Ads,Chrome,F,38,1.0,1,1,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
2,14,Ads,Chrome,F,38,1.0,2,2,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
3,14,Ads,Chrome,F,38,1.0,3,3,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
4,14,Ads,Chrome,F,38,1.0,4,4,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798


In [73]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120889 entries, 0 to 120888
Data columns (total 26 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   purchase_value                120889 non-null  int64  
 1   source                        120889 non-null  object 
 2   browser                       120889 non-null  object 
 3   sex                           120889 non-null  object 
 4   age                           120889 non-null  int64  
 5   time_velocity                 120889 non-null  float64
 6   ip_user_share_count           120889 non-null  int64  
 7   device_user_share_count       120889 non-null  int64  
 8   day                           120889 non-null  int64  
 9   hour                          120889 non-null  int64  
 10  age_manual_binned             120889 non-null  int64  
 11  purchase_value_manual_binned  120889 non-null  int64  
 12  source_Ads                    120889 non-nul

In [74]:
def run_mlflow_experiement(
    run_name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    fit_kwargs=None
):
    if fit_kwargs is None:
        fit_kwargs = {}

    with mlflow.start_run(run_name=run_name):

        # log params
        mlflow.log_params(model.get_params())

        # train
        model.fit(X_train, y_train, **fit_kwargs)

        # predictions
        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = model.decision_function(X_test)

        # metrics (safe guard optional)
        mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
        mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, y_prob))
        mlflow.log_metric("test_pr_auc", average_precision_score(y_test, y_prob))

        # report
        report = classification_report(y_test, y_pred)
        mlflow.log_text(report, "test_classification_report.txt")

        # confusion matrix
        fig1 = ConfusionMatrixDisplay.from_predictions(
            y_test,
            y_pred,
            normalize="true",
            cmap="Blues",
            values_format=".2f"
        ).figure_

        mlflow.log_figure(fig1, "confusion_matrix.png")
        plt.close(fig1)

        # ROC
        fig2 = RocCurveDisplay.from_predictions(y_test, y_prob).figure_
        mlflow.log_figure(fig2, "roc_curve.png")
        plt.close(fig2)

        # PR
        fig3 = PrecisionRecallDisplay.from_predictions(y_test, y_prob).figure_
        mlflow.log_figure(fig3, "pr_curve.png")
        plt.close(fig3)

        # log model
        mlflow.sklearn.log_model(model, "model")

In [76]:
FEATURE_SETS = {
    "ohe_full": {
        "use_cols": [
            "time_velocity",
            "ip_user_share_count", "device_user_share_count",
            "day", "hour",
            "source_Ads", "source_Direct", "source_SEO",
            "browser_Chrome", "browser_FireFox", "browser_IE", "browser_Opera", "browser_Safari",
            "sex_F", "sex_M",
            "age_manual_binned", "purchase_value_manual_binned"
        ]
    },

    "freq_enc": {
        "use_cols": [
            "time_velocity",
            "ip_user_share_count", "device_user_share_count",
            "day", "hour",
            "source_fr_enc", "browser_fr_enc",
            "sex_F", "sex_M",
            "age_manual_binned", "purchase_value_manual_binned"
        ]
    },

    "target_enc": {
        "use_cols": [
            "time_velocity",
            "ip_user_share_count", "device_user_share_count",
            "day", "hour",
            "source_targ_enc", "browser_targ_enc",
            "sex_F", "sex_M",
            "age_manual_binned", "purchase_value_manual_binned"
        ]
    },
    "no_binning_with_ohe_full": {
        "use_cols": [
            "purchase_value", "age", "time_velocity",
            "ip_user_share_count", "device_user_share_count",
            "day", "hour",
            "source_Ads", "source_Direct", "source_SEO",
            "browser_Chrome", "browser_FireFox", "browser_IE", "browser_Opera", "browser_Safari",
            "sex_F", "sex_M", 
        ]
    },
    "no_binning_with_freq_enc": {
        "use_cols": [
            "purchase_value", "age", "time_velocity",
            "ip_user_share_count", "device_user_share_count",
            "day", "hour",
            "source_fr_enc", "browser_fr_enc",
            "sex_F", "sex_M", 
        ]
    },

    "no_binning_with_target_enc": {
        "use_cols": [
            "purchase_value", "age", "time_velocity",
            "ip_user_share_count", "device_user_share_count",
            "day", "hour",
            "source_targ_enc", "browser_targ_enc",
            "sex_F", "sex_M", 
        ]
    }
}

In [77]:
def build_features(df, config):
    df_out = df.copy()

    if "use_cols" in config:
        return df_out[config["use_cols"]]

    if "drop_cols" in config:
        return df_out.drop(columns=config["drop_cols"])

    return df_out

In [78]:
def run_experiment(model_name, feature_set_name, model, X_train, y_train, X_test, y_test, extra_kwargs=None):
    
    run_name = f"{model_name}__{feature_set_name}"

    run_mlflow_experiement(
        run_name=run_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        fit_kwargs=extra_kwargs
    )

In [79]:
def compute_scale_pos_weight(y):
    y = y.squeeze()
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    return neg / pos if pos != 0 else 1

In [80]:
scale_pos_weight = compute_scale_pos_weight(y_train)
for feature_name, feature_cfg in FEATURE_SETS.items():

    X_train_fs = build_features(X_train, feature_cfg)
    X_test_fs = build_features(X_test, feature_cfg)

    # models
    models = {
        "DecisionTree": DecisionTreeClassifier(class_weight="balanced", random_state=42),

        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),

        "LGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=42
        )
    }

    for model_name, model in models.items():

        if model_name in ["XGBoost", "LGBM"]:
            split_idx = int(len(X_train_fs) * 0.8)

            train_X = X_train_fs.iloc[:split_idx]
            train_y = y_train.iloc[:split_idx].squeeze()

            val_X = X_train_fs.iloc[split_idx:]
            val_y = y_train.iloc[split_idx:].squeeze()

            lgbm_kwargs = {
                "eval_set": [(train_X, train_y), (val_X, val_y)]
            }

            run_experiment(
                model_name,
                feature_name,
                model,
                train_X,
                train_y,
                X_test_fs,
                y_test.squeeze(),
                extra_kwargs=lgbm_kwargs
            )

        else:
            run_experiment(
                model_name,
                feature_name,
                model,
                X_train_fs,
                y_train,
                X_test_fs,
                y_test
            )

2026/06/13 12:23:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/13 12:23:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetectio

🏃 View run DecisionTree__ohe_full at: http://localhost:5000/#/experiments/1/runs/e4864ac033124800a4b55ce457b66ce0
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/06/13 12:23:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed whe

🏃 View run RandomForest__ohe_full at: http://localhost:5000/#/experiments/1/runs/925c3bb7a90a4055b5d079b71a85c806
🧪 View experiment at: http://localhost:5000/#/experiments/1
[0]	validation_0-logloss:0.72577	validation_1-logloss:0.73914
[1]	validation_0-logloss:0.70067	validation_1-logloss:0.71588
[2]	validation_0-logloss:0.67855	validation_1-logloss:0.69477
[3]	validation_0-logloss:0.65705	validation_1-logloss:0.67488
[4]	validation_0-logloss:0.63824	validation_1-logloss:0.65692
[5]	validation_0-logloss:0.61936	validation_1-logloss:0.63939
[6]	validation_0-logloss:0.60221	validation_1-logloss:0.62347
[7]	validation_0-logloss:0.58616	validation_1-logloss:0.60857
[8]	validation_0-logloss:0.57119	validation_1-logloss:0.59463
[9]	validation_0-logloss:0.55734	validation_1-logloss:0.58183
[10]	validation_0-logloss:0.54453	validation_1-logloss:0.57000
[11]	validation_0-logloss:0.53225	validation_1-logloss:0.55859
[12]	validation_0-logloss:0.52076	validation_1-logloss:0.54795
[13]	validation_0

2026/06/13 12:25:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/13 12:25:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 12:26:10 WARNING mlfl

...

🏃 View run XGBoost__no_binning_with_target_enc at: http://localhost:5000/#/experiments/1/runs/1409d8d97cd24729bf9b639a977e1203
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/06/13 12:46:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/13 12:46:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetectio

[LightGBM] [Info] Number of positive: 11672, number of negative: 85039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004994 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 532
[LightGBM] [Info] Number of data points in the train set: 96711, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120689 -> initscore=-1.985917
[LightGBM] [Info] Start training from score -1.985917


2026/06/13 12:46:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during lightgbm autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 6a16a9b80dad4b86986366bf37f8fd2e. Failed operations: [RestException("INVALID_PARAMETER_VALUE: Changing param values is not allowed. Params were already logged=\'[{\'key\': \'objective\', \'old_value\': \'None\', \'new_value\': \'binary\'}]\' for run ID=\'6a16a9b80dad4b86986366bf37f8fd2e\'.")]')]
2026/06/13 12:46:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a sche

🏃 View run LGBM__no_binning_with_target_enc at: http://localhost:5000/#/experiments/1/runs/6a16a9b80dad4b86986366bf37f8fd2e
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Logistic Regression

In [63]:
X_train_lin = X_train_only_enc.drop(columns=["source_Ads", "browser_Chrome", "browser_Chrome", "sex_F"])
X_test_lin = X_test_only_enc.drop(columns=["source_Ads", "browser_Chrome", "browser_Chrome", "sex_F"])

In [64]:
X_train_lin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120889 entries, 0 to 120888
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   purchase_value           120889 non-null  int64  
 1   age                      120889 non-null  int64  
 2   time_velocity            120889 non-null  float64
 3   ip_user_share_count      120889 non-null  int64  
 4   device_user_share_count  120889 non-null  int64  
 5   day                      120889 non-null  int64  
 6   hour                     120889 non-null  int64  
 7   source_Direct            120889 non-null  bool   
 8   source_SEO               120889 non-null  bool   
 9   browser_FireFox          120889 non-null  bool   
 10  browser_IE               120889 non-null  bool   
 11  browser_Opera            120889 non-null  bool   
 12  browser_Safari           120889 non-null  bool   
 13  sex_M                    120889 non-null  bool   
dtypes: b

In [65]:
lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)
run_mlflow_experiement("LogisticRegression_Balanced", lr, X_train_lin, y_train, X_test_lin, y_test)


2026/06/12 02:09:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y wa

🏃 View run LogisticRegression_Balanced at: http://localhost:5000/#/experiments/1/runs/3f6be8dcc90a423a9bc16302aea6a957
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Decision Tree

In [49]:
dt = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)
run_mlflow_experiement("DecisionTree_Balanced", dt, X_train_only_enc, y_train, X_test_only_enc, y_test)

2026/06/12 01:53:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/12 01:54:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetectio

🏃 View run DecisionTree_Balanced at: http://localhost:5000/#/experiments/1/runs/fc6dd7309f21444b8e9c0aed0408cccd
🧪 View experiment at: http://localhost:5000/#/experiments/1


# Random Forest

In [50]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
run_mlflow_experiement("RandomForestClassifier_Balanced", rf, X_train_only_enc, y_train, X_test_only_enc, y_test)    

2026/06/12 01:55:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed whe

🏃 View run RandomForestClassifier_Balanced at: http://localhost:5000/#/experiments/1/runs/39a315dd50884506b76fb556cda39ae0
🧪 View experiment at: http://localhost:5000/#/experiments/1


# XGBoost

In [56]:

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
run_mlflow_experiement("XGBoost_Balanced", xgb, X_train_only_enc, y_train, X_test_only_enc, y_test)    

2026/06/12 02:01:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/12 02:01:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/12 02:01:37 WARNING mlfl

🏃 View run XGBoost_Balanced at: http://localhost:5000/#/experiments/1/runs/c0cda720de96498385ee4b936a269b65
🧪 View experiment at: http://localhost:5000/#/experiments/1


# LightGBM

In [67]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42
) 

split_idx = int(len(X_train_only_enc) * 0.8)

train_X = X_train_only_enc.iloc[:split_idx]
train_y = y_train.iloc[:split_idx].squeeze()   

val_X = X_train_only_enc.iloc[split_idx:]
val_y = y_train.iloc[split_idx:].squeeze()
 
lgbm_kwargs = {
    "eval_set": [(train_X, train_y), (val_X, val_y)]
}

run_mlflow_experiement(
    run_name="LGBM_Balanced", 
    lr=lgbm, 
    X_train=train_X,              # Train on the 80% split
    y_train=train_y,              # Train on the 80% split
    X_test=X_test_only_enc,       # Test on the clean holdout test set
    y_test=y_test.squeeze(),      # Squeezed to maintain 1D shape consistency
    kwargs=lgbm_kwargs
)

2026/06/12 02:15:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/12 02:15:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetectio

[LightGBM] [Info] Number of positive: 11672, number of negative: 85039
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003602 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 540
[LightGBM] [Info] Number of data points in the train set: 96711, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120689 -> initscore=-1.985917
[LightGBM] [Info] Start training from score -1.985917


2026/06/12 02:15:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/12 02:15:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/12 02:15:27 WARNING mlfl

🏃 View run LGBM_Balanced at: http://localhost:5000/#/experiments/1/runs/f55522f770af494c90a4dd8f8e984254
🧪 View experiment at: http://localhost:5000/#/experiments/1


# Save CSV

In [ ]:
# X_train_resampled.to_csv(PROCESSED_DATA_PATH / "5.1_X_train.csv", index=False)
# y_train_resampled.to_csv(PROCESSED_DATA_PATH / "5.2_y_train.csv", index=False)
